# Flash attention
## 搭建环境
在ubuntu虚拟机上搭建了环境，可以正确编译以及cuda程序，实现cuda程序的自动跳转，方便编写cuda代码
因此编写代码就放在虚拟机上

因为flash attention2 3 需要利用特殊的硬件特性，因此需要在租相应的平台，在平台上搭建环境很麻烦

先看看autodl，能不能采用docker的方式运行
## 实现flash attention
### 实现层次
需要先看看怎么实现，应该是直接实现flash attention 算子，包括对应的forward、backward

上层的transfomer 模块相应的调用算子（看一下以前的transformer实现）

最后确认了实现的层次：Ops -> CUDA Kernel -> Pybind -> Op Wrapper -> Module Call
### 实现方法
应该用什么实现呢？cuda、triton、cuTile

使用cuda编程的话，应该借用cutblass模版库进行编程

如果使用triton实现的话，绕过了pybind，但是将cuda后端管理的ptr给triton模块，然后再在triton层面进行编程

使用cuTile编程的话，依旧是python编程，和triton类似

### 测试程序
需要先编写测试程序，首先是flash attention 算子的测试程序，该测试程序需要包括一下几个方面

1.直接调用cuda程序中的实现进行测试（确定在ops层面进行测试）

2.需要验证算子的正确性，那和什么进行比对呢（或者在python层面进行比对正确性，参考一下以前的比较）

3.每个测试应该进行多次迭代，从而可以测试时间，包括不同seq长度

（可以参考官方的flash attention实现，看看它是怎么进行测试的）





接下来要做的事：

在autodl平台上跑一次

了解cuTile，决定是用cutlass还是用cuTile

看看别人的实现以及怎么测试、怎么benchmark

backward 不应该都是tensor操作吗，直接调用Narray api不就构建不了计算图了吗？（需要构建一个ops用来计算backward）

测试的时候是调用ops测试，还是module呢，以及怎么获得时间


In [ ]:
!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

# Download the PTB dataset

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

In [ ]:
!make clean
!make

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## 测试
### 正确性验证
如果在NDArray层面去测试的话，需要借用numpy来进行比较，而numpy没有现有的api计算flash attentionn，需要根据已知的qkv计算self attention，因此需要创建numpy的qkv，然后根据self attention的定义去计算，再与cuda 的NDArray计算结果进行比较，比较麻烦。

因此本测试在flash attention module层面进行正确性验证，仿照已有测试中的attention_activation 测试编写，将flash attention的结果与已知的label进行对比，同时编写了新的测试将结果和torch的flash attention 计算结果对比，在与torch对比的测试用例 sequence length比较长，符合实际。
### benchmark
对于benchmark说，需要计算TFLOPS，为了最贴近计算，采用算子层面进行benchmark，先对GPU进行预热，flashattention进行计算，迭代50次，计算时间以及总的操作数，从而计算TFOPS。当前的计算基于causal = false，dropout = 0

In [ ]:
!python3 -m pytest tests/hw4/test_transformer.py -l -v -k "attention_activation_vs_torch"

In [ ]:
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and 64 and False and 0.0 and cuda"

In [ ]:
!python3 -m pytest tests/project/test_flashattention.py  -s -l -v -k "test_attention_activation_vs_torch and 5 and 1024 and 64 and False and 0.0 and cuda" >> debug.txt

In [ ]:
# stub hook 验证
import sys
!{sys.executable} -m pytest tests/project/test_flashattention_stub.py -l -v

In [ ]:
# 计算TFLOPS
!python3 tests/project/benchmark.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# baseline
!python3 tests/project/benchmark_torch.py --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1

## stub
对于ops层面的stub来说，是返回tensor tuple还是tensor呢？
* 需要查看probs对于后续有没有什么作用，如果没有作用可以直接返回result tensor
* 如果返回tensor tuple，是backend计算直接返回tensor tuple还是在ops层进行组装后返回，需要参考stack的实现，返回tensor tuple后进行第i个结果的取用会不会产生额外的开销

最后决定只是返回result，因此定义为tensorops


## 调用层次关系
module调用ops进行计算，ops的forward计算的是NDArray，使用NDArray定义的array api进行计算，NDArray的backend device不同，因此调用进不同的device function。

## 精度
在pytorch中的attention的实现中，因为A100的tensor core只接受输入的形式为 fp16/bf16/tf32，不支持fp32的输入，因此没有开启混合精度的情况下，是没有使用flash attention的实现的，而使用的是原生的的其他优化过的kernel

如果开启混合精度模式，计算的流程如下：QKV（fp32）加载到sram中cast 为（fp16），使用tensor做矩阵乘法时，结果accumulator保存为fp32。进行softmax的时候，为了防止溢出保持fp32，再cast为fp16与 V（fp16）相乘，
最后的结果O为fp16（为了节省sram 写到 GMEM中的带宽）。

现在我自己的实现最好确定输入的类型时fp16，那这样其他实现的cuda计算就用不了了，一个weired解决办法是输入继续保持为fp32但是计算的结果o为fp32再重新cast为fp16。

* 看看别人的flash attention怎么实现的
    * 别人的flash attention中没有使用cute模版编程，只是借用了cutlass的gemm实现来优化自己的设计。

* 如果在我的实现中使用cutlass，应该怎么使用，和cute有区别吗？
    * CuTe 强大的 Layout Algebra (布局代数) 能够让你在不陷入指针算术泥潭的情况下，优雅地处理复杂的 Tensor Core 数据映射、Shared Memory Swizzle 和 Bank Conflict。


1. 现在需要研究cute 的核心，使用cute 模版编程来实现自己的flash attention。




## flash attention with mma 
在navie 实现上做了这些优化：
1. 传输优化
* global -> share memory的传输使用了指令 **cp.async.ca.shared.global.L2::128B** ,该指令每个线程每次传输128bit的数据，因此传输的数据也需要满足相应的要求，即如果传输的数据是fp16(m, n)，在n维度上对应的数据物理地址要求每8个元素一组连续，8个元素与另一组则没有要求，且他是异步传输指令。
* 同时对于线程的排布采用了**Layout<Shape<_16, _8>, Stride<_8, _1>>** row major的方法，因为global memory中源数据是row major储存的，这样处理的话，多个线程访问global memory的transaction会合并为一个transaction。
2. 矩阵计算优化
* matrix计算从最原始的使用FMA改为了使用**mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32**，与FMA不同的是，这里的矩阵乘法是以warp为单位进行计算的，每个thread register内需要包含A B C D相应的fragment。在make tiled mma的时候，第二个参数thr_layout，如果在K维度上有排列，举出这样一个例子便于理解：（Layout<Shape<_2, _1, _2>>{}）调用gemm后，那么对于(m, n)的结果矩阵C，前1/2 K维度的结果保留在warp 0、1，后1/2 K维度的结果保留在warp 2、3中，因此需要再进行一次规约才能得到最后的结果矩阵。
* gemm api调用时，需要满足这样的假设：A、B的逻辑形状分别是(m, k) (n, k)，这样gemm才知道分得清维度m、n、k。因此在计算mma2时，对sV进行了转置再传给gemm。至于对于采用的atom **SM80_16x8x8_F32F16F16F32_TN** 最后面是T还是N，其实和逻辑矩阵的形状没有关系，这里的TN影响的是最后thread中fragment到底是哪一部分，cute TV value为我们处理了。
3. atom -> tiled -> all（以mma举例解释，copy指令差不多）
* atom 即一个warp指令，通过make tiled，使用多个warp计算或者一个warp计算更多的值形成tiled mma。
* 一个tiled mma对A、B、C进行partition，对于结果矩阵来说使用tiled mma的逻辑矩阵（m，n）平铺整个大的（M，N），即在row、column方向上分别进行除法，产生了新的维度。这里的底层逻辑还是通过一个warp计算更多的值实现的。
4. 如何实现不需要share memory的过渡，mma1储存在register中的accumulator C直接用于mma2的operator A？
* 每个register是每个thread私有的，因此这就要求形成tiled mma最后通过partition形成的fragment，对于每个thread来说拥有的mma2 accumulator C的值刚好可以用于mma1 operator A。
* 第一层即选用atom，选用的atom满足**ALayout = CLayout**，这样在一个atom内部，mma1的C fragment即mma2的A fragment
* 第二层即形成tiled mma，mma1 C和mma2 A共有的维度是m，线程的排布必须都集中在该维度上，否则就会出现fragment大小不一致的情况
* 第三层即partition，对于partition的切分来说，满足上面的两个条件后，切分后的fragment相同是自然的，因为切分即一个warp计算更多的值实现，而且切分的逻辑都是对逻辑矩阵中的row column进行切分
5. 对于不采用share memory过渡P的方法，如何计算row max以及row sum呢？
* fragment是thread私有的，以及一个thread的数据跨越不同的row，以及一个row的数据包含在多个thread中，这就要求我们清晰的明白TV layout，通过print_latex可以得到tiled_mma的(m, n) -> (T, V)，从而推断出thread的fragment
* 因为一个row的数据储存在相同warp下的多个thread中，因此采用指令__shfl_xor_sync进行规约，最后的结果是每个thread都有他的那部分row的row max和row sum，因为mma1 fragment C = mma2 fragment A = mma2 fragment C的递进关系，这里的row max又可以用来rescale output O


还可以做的优化：
1. share memory -> register 采用ldmatrix
2. share memory swizzled
3. 最后在register中的output直接copy到global中，或者lshare memory再copy到global中怎么优化
4. software pipeline，怎么进行copy和compute overlap
